# Single Time MetaUncertaintyGate

Colab notebook version of `single time.py`. It trains and evaluates only the custom MetaUncertaintyGate model on the single-pass JSONL data. Baseline models are intentionally not included.

Default data source: `https://raw.githubusercontent.com/hsun0912/5703sunhao/main/data/single%20time.jsonl`.


In [ ]:
# Colab/runtime check
import sys
print("Python:", sys.version)
try:
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
except Exception as exc:
    print("Torch import failed:", exc)
try:
    import numpy as np
    print("NumPy:", np.__version__)
except Exception as exc:
    print("NumPy import failed:", exc)


## Model And Helper Definitions

Run the next cells once to define the single-pass schema, model, training loop, metrics, and experiment runner.


In [ ]:
"""
Single-pass MetaUncertaintyGate runner.

This script is for the v13 single-pass feature file generated from one LLM
answer per prompt. It intentionally evaluates only the custom
MetaUncertaintyGate model. Baseline models from the notebook are not included.

Default label convention:
    1 = failure / hallucination
    0 = correct

The user-provided single-pass JSONL already contains `single_is_failure`, so
that is the default target.
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
import random
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    import numpy as np
except ModuleNotFoundError as exc:
    raise SystemExit(
        "Missing dependency: numpy. Install numpy before running this script."
    ) from exc

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise SystemExit(
        "Missing dependency: torch. Run this script in the same Python/Colab "
        "environment used for the previous MetaUncertaintyGate notebooks."
    ) from exc


DEFAULT_DATA_PATH = (
    r"C:\Users\Administrator\Documents\xwechat_files"
    r"\wxid_tsjr7qv0equ911_c6df\msg\file\2026-05"
    r"\Feature_Extraction_v13_2000balanced_singlepass_from_k5.jsonl"
)
DEFAULT_OUTPUT_DIR = os.path.join("outputs", "single_time")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# =============================================================================
# Single-pass branch schema


In [ ]:
# =============================================================================

# Entropy/probability features available from one generated answer.
BRANCH_ENTROPY_KEYS = [
    "sp_entropy_mean",
    "sp_entropy_max",
    "sp_entropy_last",
    "sp_entropy_gap",
    "sp_avg_token_logprob",
    "sp_token_logprob_std",
    "sp_high_entropy_token_ratio",
]

# The k=5 hidden-state geometry branch cannot be reproduced from one answer.
# Keep the one available hidden representation magnitude as a small geometry
# branch instead of silently filling the missing PCA/eccentricity features.
BRANCH_GEOMETRY_KEYS = [
    "sp_hidden_norm",
]

# Single-answer behavioral and answer-shape signals.
BRANCH_BEHAVIOR_KEYS = [
    "sp_generated_token_len",
    "sp_ended_by_eos",
    "sp_max_token_hit",
    "sp_answer_repetition_rate",
    "sp_raw_answer_word_len",
    "sp_raw_answer_token_len",
    "sp_processed_answer_word_len",
    "sp_has_final_answer",
    "sp_is_mmlu_option",
]

# Context remains gate-only. These features can route evidence but do not
# directly contribute a branch failure logit.
BRANCH_CONTEXT_KEYS = [
    "question_token_len",
    "prompt_token_len",
    "q_is_math_keyword",
    "q_is_code_keyword",
    "q_has_number",
    "q_has_option_marker",
    "q_has_why",
    "q_has_how",
    "q_has_what",
    "q_has_calculation_word",
    "lang_en",
    "lang_zh",
    "lang_other",
    "lang_unknown",
    "task_factual_qa",
    "task_math_reasoning",
    "task_multiple_choice_qa",
    "dataset_difficulty_prior",
]

ALL_BRANCH_GROUPS = [
    BRANCH_ENTROPY_KEYS,
    BRANCH_GEOMETRY_KEYS,
    BRANCH_BEHAVIOR_KEYS,
    BRANCH_CONTEXT_KEYS,
]
BRANCH_NAMES = ["Entropy", "Geometry", "Behavioral", "Context"]
BRANCH_DIMS = [len(g) for g in ALL_BRANCH_GROUPS]
CONTEXT_GROUP_IDX = 3
CONTEXT_DIM = len(BRANCH_CONTEXT_KEYS)
EVIDENCE_BRANCH_IDXS = [0, 1, 2]
EVIDENCE_BRANCH_NAMES = [BRANCH_NAMES[i] for i in EVIDENCE_BRANCH_IDXS]

K5_FEATURES_DELETED_FOR_SINGLE_PASS = [
    "entropy_mean_avg",
    "entropy_mean_std",
    "entropy_max_avg",
    "entropy_max_std",
    "entropy_last_avg",
    "entropy_last_std",
    "entropy_gap",
    "eccentricity_mean",
    "eccentricity_max",
    "pca_var_pc1",
    "pca_var_pc2",
    "pca_var_pc3",
    "pca_var_top3_sum",
    "hidden_norm_mean",
    "hidden_norm_std",
    "self_consistency",
    "answer_len_mean",
    "answer_len_std",
    "answer_unique_ratio",
    "ds_truthfulqa",
    "ds_gsm8k",
    "ds_mmlu",
]

DATASET_NAME_TO_ID = {
    "GSM8K": 0,
    "MMLU": 1,
    "TruthfulQA": 2,
}


In [ ]:
# =============================================================================
# Utilities


In [ ]:
# =============================================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def as_float(value: Any, default: float = 0.0) -> float:
    try:
        x = float(value)
    except (TypeError, ValueError):
        return default
    return x if math.isfinite(x) else default


def json_convert(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_convert(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_convert(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        v = float(value)
        return v if math.isfinite(v) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    return value


def save_json(data: Dict[str, Any], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_convert(data), f, ensure_ascii=False, indent=2)


def print_feature_schema() -> None:
    print("\n" + "=" * 72)
    print("Single-pass feature schema used by this runner")
    print("=" * 72)
    for name, keys in zip(BRANCH_NAMES, ALL_BRANCH_GROUPS):
        role = "gate-only" if name == "Context" else "evidence"
        print(f"  {name:<10} ({role:<9}) dim={len(keys):>2}: {', '.join(keys)}")
    print("-" * 72)
    print("Deleted k=5-only features:")
    for key in K5_FEATURES_DELETED_FOR_SINGLE_PASS:
        print(f"  - {key}")
    print("=" * 72)


In [ ]:
# =============================================================================
# Data loading


In [ ]:
# =============================================================================

def _resolve_failure_label(
    obj: Dict[str, Any],
    label_source: str,
    label_mode: str,
) -> int:
    source = label_source
    if source == "auto":
        if obj.get("single_is_failure") is not None:
            source = "single_is_failure"
        elif obj.get("single_is_correct") is not None:
            source = "single_is_correct"
        else:
            source = "label"

    if source == "single_is_failure":
        value = obj.get("single_is_failure")
        if value is None:
            raise ValueError("single_is_failure is missing")
        y = int(value)
        if y not in (0, 1):
            raise ValueError(f"single_is_failure must be 0/1, got {value!r}")
        return y

    if source == "single_is_correct":
        value = obj.get("single_is_correct")
        if value is None:
            raise ValueError("single_is_correct is missing")
        y_correct = int(value)
        if y_correct not in (0, 1):
            raise ValueError(f"single_is_correct must be 0/1, got {value!r}")
        return 1 - y_correct

    if source != "label":
        raise ValueError(f"unknown label_source={label_source!r}")

    raw = obj.get("label")
    if raw is None:
        raise ValueError("label is None")
    raw_label = int(raw)
    if raw_label not in (0, 1):
        raise ValueError(f"label must be 0/1, got {raw!r}")

    mode = label_mode
    if mode == "auto":
        if obj.get("single_is_correct") is not None:
            mode = "correct_is_1"
        elif obj.get("single_is_failure") is not None:
            mode = "failure_is_1"
        else:
            mode = "correct_is_1"

    if mode == "correct_is_1":
        return 1 - raw_label
    if mode == "failure_is_1":
        return raw_label
    raise ValueError(f"unknown label_mode={label_mode!r}")


def load_jsonl(
    path: str,
    label_source: str = "single_is_failure",
    label_mode: str = "auto",
) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    total = 0
    skipped = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            total += 1
            line = line.strip()
            if not line:
                skipped["empty"] += 1
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                skipped["malformed"] += 1
                continue

            features = obj.get("feature_dict", obj.get("features"))
            if not isinstance(features, dict):
                skipped["no_features"] += 1
                continue

            try:
                label = _resolve_failure_label(obj, label_source, label_mode)
            except (TypeError, ValueError):
                skipped["bad_label"] += 1
                continue

            records.append(
                {
                    "features": features,
                    "label": int(label),
                    "dataset": obj.get("dataset", "unknown"),
                    "task_type": obj.get("task_type", "unknown"),
                    "domain": obj.get("domain", "unknown"),
                    "sample_id": obj.get("sample_id", f"line_{line_no}"),
                    "raw_label": obj.get("label"),
                    "single_is_correct": obj.get("single_is_correct"),
                    "single_is_failure": obj.get("single_is_failure"),
                    "majority_is_correct": obj.get("majority_is_correct"),
                    "raw_single_answer": obj.get("raw_single_answer", ""),
                    "processed_single_answer": obj.get("processed_single_answer", ""),
                }
            )

    print("\n" + "=" * 72)
    print("Data loading summary")
    print("=" * 72)
    print(f"  Path              : {path}")
    print(f"  Total lines       : {total}")
    print(f"  Loaded records    : {len(records)}")
    print(f"  Skipped records   : {sum(skipped.values())}")
    if skipped:
        for key in sorted(skipped):
            print(f"    {key:<14}: {skipped[key]}")
    print(f"  Label source      : {label_source}")
    print(f"  Label mode        : {label_mode}")
    print("  Training label    : 1=failure, 0=correct")
    print("=" * 72)

    return records


def validate_features(records: List[Dict[str, Any]], allow_missing: bool = False) -> Dict[str, Any]:
    required = [key for group in ALL_BRANCH_GROUPS for key in group]
    missing_counts = {key: 0 for key in required}
    nonfinite_counts = {key: 0 for key in required}
    observed = Counter()

    for record in records:
        features = record["features"]
        observed.update(features.keys())
        for key in required:
            if key not in features:
                missing_counts[key] += 1
            elif not math.isfinite(as_float(features[key], default=float("nan"))):
                nonfinite_counts[key] += 1

    missing = {k: v for k, v in missing_counts.items() if v > 0}
    nonfinite = {k: v for k, v in nonfinite_counts.items() if v > 0}
    unused_observed = sorted(set(observed) - set(required))

    print("\n" + "=" * 72)
    print("Feature validation")
    print("=" * 72)
    print(f"  Required features : {len(required)}")
    print(f"  Observed features : {len(observed)}")
    print(f"  Missing required  : {len(missing)}")
    print(f"  Non-finite values : {sum(nonfinite.values())}")
    if unused_observed:
        print(f"  Unused observed   : {', '.join(unused_observed)}")
    if missing:
        for key, count in sorted(missing.items()):
            print(f"    missing {key}: {count}/{len(records)}")
    if nonfinite:
        for key, count in sorted(nonfinite.items()):
            print(f"    nonfinite {key}: {count}/{len(records)}")
    print("=" * 72)

    if missing and not allow_missing:
        raise ValueError(
            "Required single-pass features are missing. Use --allow-missing-features "
            "only if you intentionally want missing values filled with 0.0."
        )

    return {
        "required_features": required,
        "observed_features": sorted(observed),
        "unused_observed_features": unused_observed,
        "missing_counts": missing,
        "nonfinite_counts": nonfinite,
    }


def print_data_stats(records: List[Dict[str, Any]]) -> None:
    total = len(records)
    failures = sum(int(r["label"]) for r in records)
    correct = total - failures
    by_dataset: Dict[str, Dict[str, int]] = defaultdict(lambda: {"total": 0, "fail": 0})
    by_task: Dict[str, Dict[str, int]] = defaultdict(lambda: {"total": 0, "fail": 0})

    for r in records:
        by_dataset[str(r["dataset"])]["total"] += 1
        by_dataset[str(r["dataset"])]["fail"] += int(r["label"])
        by_task[str(r["task_type"])]["total"] += 1
        by_task[str(r["task_type"])]["fail"] += int(r["label"])

    print("\n" + "=" * 72)
    print("Label distribution")
    print("=" * 72)
    print(f"  ALL: total={total} correct={correct} failure={failures} "
          f"failure_rate={failures / max(total, 1):.3f}")
    print("\n  By dataset:")
    for name in sorted(by_dataset):
        item = by_dataset[name]
        fail = item["fail"]
        ok = item["total"] - fail
        print(f"    {name:<16} total={item['total']:>5} correct={ok:>5} "
              f"failure={fail:>5} failure_rate={fail / max(item['total'], 1):.3f}")
    print("\n  By task:")
    for name in sorted(by_task):
        item = by_task[name]
        fail = item["fail"]
        ok = item["total"] - fail
        print(f"    {name:<24} total={item['total']:>5} correct={ok:>5} "
              f"failure={fail:>5} failure_rate={fail / max(item['total'], 1):.3f}")
    print("=" * 72)


In [ ]:
# =============================================================================
# Metrics without sklearn


In [ ]:
# =============================================================================

def _average_ranks(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(len(values), dtype=np.float64)
    i = 0
    while i < len(values):
        j = i + 1
        while j < len(values) and values[order[j]] == values[order[i]]:
            j += 1
        avg_rank = 0.5 * (i + 1 + j)
        ranks[order[i:j]] = avg_rank
        i = j
    return ranks


def binary_auroc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=np.float64)
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = _average_ranks(y_score)
    rank_sum_pos = float(ranks[y_true == 1].sum())
    return (rank_sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def binary_auprc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=np.float64)
    n_pos = int((y_true == 1).sum())
    if n_pos == 0:
        return float("nan")
    order = np.argsort(-y_score, kind="mergesort")
    y_sorted = y_true[order]
    tp = np.cumsum(y_sorted == 1)
    precision = tp / np.arange(1, len(y_sorted) + 1)
    return float(precision[y_sorted == 1].sum() / n_pos)


def compute_ece(y_true: np.ndarray, y_score: np.ndarray, n_bins: int = 10) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_score = np.asarray(y_score, dtype=np.float64)
    if len(y_true) == 0:
        return float("nan")
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi >= 1.0:
            mask = (y_score >= lo) & (y_score <= hi)
        else:
            mask = (y_score >= lo) & (y_score < hi)
        if int(mask.sum()) == 0:
            continue
        conf = float(y_score[mask].mean())
        acc = float(y_true[mask].mean())
        ece += (float(mask.mean()) * abs(conf - acc))
    return float(ece)


def classification_from_preds(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    n = len(y_true)
    if n == 0:
        return {
            "n": 0,
            "accuracy": float("nan"),
            "precision": float("nan"),
            "recall": float("nan"),
            "f1": float("nan"),
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
            "pred_pos_rate": float("nan"),
        }

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    if math.isfinite(precision) and math.isfinite(recall) and (precision + recall) > 0:
        f1 = 2.0 * precision * recall / (precision + recall)
    else:
        f1 = float("nan")
    return {
        "n": int(n),
        "accuracy": float((tp + tn) / n),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "pred_pos_rate": float((y_pred == 1).mean()),
    }


def classification_from_scores(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float = 0.5,
) -> Dict[str, Any]:
    y_pred = (np.asarray(y_score) >= float(threshold)).astype(int)
    out = classification_from_preds(y_true, y_pred)
    out["threshold"] = float(threshold)
    return out


def compute_all_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float = 0.5,
) -> Dict[str, Any]:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=np.float64)
    out = {
        "auroc": binary_auroc(y_true, y_score),
        "auprc": binary_auprc(y_true, y_score),
        "brier": float(np.mean((y_score - y_true) ** 2)) if len(y_true) else float("nan"),
        "ece": compute_ece(y_true, y_score),
        "label_pos_rate": float((y_true == 1).mean()) if len(y_true) else float("nan"),
    }
    out.update(classification_from_scores(y_true, y_score, threshold))
    return out


def _score_for_metric(metrics: Dict[str, Any], metric: str) -> float:
    if metric == "f1":
        return float(metrics["f1"])
    if metric == "accuracy":
        return float(metrics["accuracy"])
    if metric == "balanced_acc":
        recall = float(metrics["recall"])
        tnr = metrics["tn"] / max(metrics["tn"] + metrics["fp"], 1)
        return 0.5 * (recall + tnr) if math.isfinite(recall) else float("nan")
    if metric == "youden":
        recall = float(metrics["recall"])
        fpr = metrics["fp"] / max(metrics["fp"] + metrics["tn"], 1)
        return recall - fpr if math.isfinite(recall) else float("nan")
    return float(metrics.get(metric, float("nan")))


def tune_threshold(
    y_true: np.ndarray,
    y_score: np.ndarray,
    metric: str = "f1",
    fallback: float = 0.5,
    n_grid: int = 200,
    min_pred_pos_rate: float = 0.02,
    max_pred_pos_rate: float = 0.85,
    avoid_degenerate: bool = True,
) -> Tuple[float, float]:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=np.float64)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float(fallback), float("nan")

    unique_scores = np.unique(y_score)
    if len(unique_scores) > n_grid:
        unique_scores = np.linspace(float(y_score.min()), float(y_score.max()), n_grid)
    grid = np.unique(np.clip(np.concatenate([[0.0], unique_scores, [1.0]]), 0.0, 1.0))

    best_t = float(fallback)
    best_s = -float("inf")
    best_distance = float("inf")
    any_allowed = False
    for threshold in grid:
        threshold = float(threshold)
        pred_pos_rate = float(np.mean(y_score >= threshold))
        if avoid_degenerate and (pred_pos_rate <= 0.0 or pred_pos_rate >= 1.0):
            continue
        if pred_pos_rate < min_pred_pos_rate or pred_pos_rate > max_pred_pos_rate:
            continue
        any_allowed = True
        metrics = classification_from_scores(y_true, y_score, threshold)
        score = _score_for_metric(metrics, metric)
        distance = abs(threshold - float(fallback))
        if math.isfinite(score) and (
            score > best_s + 1e-12
            or (abs(score - best_s) <= 1e-12 and distance < best_distance)
        ):
            best_s = float(score)
            best_t = float(threshold)
            best_distance = float(distance)

    if not any_allowed:
        return float(fallback), float("nan")
    return best_t, best_s


def tune_group_thresholds(
    records: Sequence[Dict[str, Any]],
    y_true: np.ndarray,
    y_score: np.ndarray,
    group_key: str,
    metric: str,
    fallback: float,
    min_n: int = 20,
    min_pred_pos_rate: float = 0.02,
    max_pred_pos_rate: float = 0.85,
    avoid_degenerate: bool = True,
) -> Dict[str, Dict[str, Any]]:
    groups: Dict[str, List[int]] = defaultdict(list)
    for i, record in enumerate(records):
        groups[str(record.get(group_key, "unknown"))].append(i)

    out: Dict[str, Dict[str, Any]] = {}
    for group_name, idxs in sorted(groups.items()):
        idx = np.asarray(idxs, dtype=int)
        group_y = y_true[idx]
        group_s = y_score[idx]
        if len(idx) < min_n or len(np.unique(group_y)) < 2:
            thr = float(fallback)
            score = float("nan")
            tuned = False
        else:
            thr, score = tune_threshold(
                group_y,
                group_s,
                metric=metric,
                fallback=fallback,
                min_pred_pos_rate=min_pred_pos_rate,
                max_pred_pos_rate=max_pred_pos_rate,
                avoid_degenerate=avoid_degenerate,
            )
            tuned = math.isfinite(score)
        out[group_name] = {
            "threshold": float(thr),
            "score": float(score),
            "n": int(len(idx)),
            "label_pos_rate": float((group_y == 1).mean()) if len(idx) else float("nan"),
            "pred_pos_rate": float((group_s >= thr).mean()) if len(idx) else float("nan"),
            "tuned": bool(tuned),
        }
    return out


def apply_group_thresholds(
    records: Sequence[Dict[str, Any]],
    y_score: np.ndarray,
    thresholds: Dict[str, Dict[str, Any]],
    group_key: str,
    fallback: float,
) -> Tuple[np.ndarray, np.ndarray]:
    preds = np.zeros(len(records), dtype=int)
    used = np.zeros(len(records), dtype=np.float64)
    for i, record in enumerate(records):
        group_name = str(record.get(group_key, "unknown"))
        threshold = float(thresholds.get(group_name, {}).get("threshold", fallback))
        used[i] = threshold
        preds[i] = int(y_score[i] >= threshold)
    return preds, used


def compute_group_metrics(
    records: Sequence[Dict[str, Any]],
    y_true: np.ndarray,
    y_score: np.ndarray,
    group_key: str,
    threshold: float = 0.5,
    group_thresholds: Optional[Dict[str, Dict[str, Any]]] = None,
) -> Dict[str, Dict[str, Any]]:
    groups: Dict[str, List[int]] = defaultdict(list)
    for i, record in enumerate(records):
        groups[str(record.get(group_key, "unknown"))].append(i)

    out: Dict[str, Dict[str, Any]] = {}
    for group_name, idxs in sorted(groups.items()):
        idx = np.asarray(idxs, dtype=int)
        if group_thresholds is not None:
            thr = float(group_thresholds.get(group_name, {}).get("threshold", threshold))
        else:
            thr = float(threshold)
        out[group_name] = compute_all_metrics(y_true[idx], y_score[idx], threshold=thr)
    return out


def mean_finite(values: Iterable[float]) -> float:
    vals = [float(v) for v in values if math.isfinite(float(v))]
    return float(np.mean(vals)) if vals else float("nan")


def worst_finite(values: Iterable[float]) -> float:
    vals = [float(v) for v in values if math.isfinite(float(v))]
    return float(min(vals)) if vals else float("nan")


def print_threshold_table(thresholds: Dict[str, Dict[str, Any]], title: str) -> None:
    print("\n" + title)
    print("  group                       n    thr     score   label+  pred+  tuned")
    print("  " + "-" * 72)
    for group, item in thresholds.items():
        print(
            f"  {group:<24} {item['n']:>5}  "
            f"{item['threshold']:>6.3f}  {item['score']:>7.3f}  "
            f"{item['label_pos_rate']:>6.3f}  {item['pred_pos_rate']:>6.3f}  "
            f"{str(item['tuned']):>5}"
        )


def print_group_metrics_table(metrics: Dict[str, Dict[str, Any]], title: str) -> None:
    print("\n" + title)
    print("  group                       n   AUROC   AUPRC   Acc     Prec    Rec     F1")
    print("  " + "-" * 82)
    for group, item in metrics.items():
        print(
            f"  {group:<24} {int(item['n']):>5}  "
            f"{item['auroc']:>6.3f}  {item['auprc']:>6.3f}  "
            f"{item['accuracy']:>6.3f}  {item['precision']:>6.3f}  "
            f"{item['recall']:>6.3f}  {item['f1']:>6.3f}"
        )


In [ ]:
# =============================================================================
# Split and normalization


In [ ]:
# =============================================================================

def stratified_split_records(
    records: List[Dict[str, Any]],
    train_ratio: float,
    val_ratio: float,
    seed: int,
    stratify_keys: Tuple[str, ...] = ("dataset", "label"),
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    rng = random.Random(seed)
    grouped: Dict[Tuple[Any, ...], List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        key_parts = []
        for key in stratify_keys:
            value = record.get(key, "unknown")
            if key == "label":
                value = int(value)
            else:
                value = str(value)
            key_parts.append(value)
        grouped[tuple(key_parts)].append(record)

    train: List[Dict[str, Any]] = []
    val: List[Dict[str, Any]] = []
    test: List[Dict[str, Any]] = []

    for _, group in sorted(grouped.items(), key=lambda kv: str(kv[0])):
        items = list(group)
        rng.shuffle(items)
        n = len(items)
        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))

        if n >= 3:
            n_train = max(1, min(n - 2, n_train))
            n_val = max(1, min(n - n_train - 1, n_val))
        elif n == 2:
            n_train, n_val = 1, 0
        else:
            n_train, n_val = n, 0

        train.extend(items[:n_train])
        val.extend(items[n_train:n_train + n_val])
        test.extend(items[n_train + n_val:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


def count_by_group(records: Sequence[Dict[str, Any]], key: str) -> Dict[str, int]:
    counts = Counter(str(r.get(key, "unknown")) for r in records)
    return dict(sorted(counts.items()))


def split_summary(
    train: Sequence[Dict[str, Any]],
    val: Sequence[Dict[str, Any]],
    test: Sequence[Dict[str, Any]],
) -> Dict[str, Any]:
    return {
        "train_n": len(train),
        "val_n": len(val),
        "test_n": len(test),
        "train_dataset": count_by_group(train, "dataset"),
        "val_dataset": count_by_group(val, "dataset"),
        "test_dataset": count_by_group(test, "dataset"),
        "train_label": count_by_group(train, "label"),
        "val_label": count_by_group(val, "label"),
        "test_label": count_by_group(test, "label"),
    }


class FeatureNormalizer:
    """Per-branch standardization fitted on the training split only."""

    def __init__(self) -> None:
        self.means: List[np.ndarray] = []
        self.stds: List[np.ndarray] = []

    def _extract_group(self, records: Sequence[Dict[str, Any]], group_idx: int) -> np.ndarray:
        keys = ALL_BRANCH_GROUPS[group_idx]
        matrix = np.zeros((len(records), len(keys)), dtype=np.float32)
        for i, record in enumerate(records):
            features = record["features"]
            for j, key in enumerate(keys):
                matrix[i, j] = as_float(features.get(key, 0.0))
        return matrix

    def fit(self, records: Sequence[Dict[str, Any]]) -> "FeatureNormalizer":
        if len(records) == 0:
            raise ValueError("Cannot fit FeatureNormalizer on an empty training set.")
        self.means = []
        self.stds = []
        for group_idx in range(len(ALL_BRANCH_GROUPS)):
            matrix = self._extract_group(records, group_idx)
            mean = matrix.mean(axis=0)
            std = matrix.std(axis=0)
            std = np.where(std < 1e-6, 1.0, std)
            self.means.append(mean.astype(np.float32))
            self.stds.append(std.astype(np.float32))
        return self

    def transform(self, records: Sequence[Dict[str, Any]]) -> List[np.ndarray]:
        if not self.means or not self.stds:
            raise RuntimeError("FeatureNormalizer.transform called before fit().")
        out: List[np.ndarray] = []
        for group_idx in range(len(ALL_BRANCH_GROUPS)):
            matrix = self._extract_group(records, group_idx)
            scaled = (matrix - self.means[group_idx]) / self.stds[group_idx]
            scaled = np.nan_to_num(scaled, nan=0.0, posinf=0.0, neginf=0.0)
            out.append(scaled.astype(np.float32))
        return out

    def state_dict(self) -> Dict[str, Any]:
        return {
            "means": [x.tolist() for x in self.means],
            "stds": [x.tolist() for x in self.stds],
            "branch_groups": ALL_BRANCH_GROUPS,
        }


def dataset_balance_weights(records: Sequence[Dict[str, Any]]) -> List[float]:
    if not records:
        return []
    counts = Counter(str(r.get("dataset", "unknown")) for r in records)
    n_groups = max(len(counts), 1)
    raw = [
        len(records) / (n_groups * counts[str(r.get("dataset", "unknown"))])
        for r in records
    ]
    mean_weight = float(np.mean(raw)) if raw else 1.0
    return [float(w / max(mean_weight, 1e-8)) for w in raw]


class MetaGateDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, Any]], normalizer: FeatureNormalizer) -> None:
        self.records = list(records)
        self.groups = normalizer.transform(records)
        self.contexts = self.groups[CONTEXT_GROUP_IDX]
        self.labels = [float(r["label"]) for r in records]
        self.weights = dataset_balance_weights(records)
        self.dataset_ids = [
            int(DATASET_NAME_TO_ID.get(str(r.get("dataset", "unknown")), -1))
            for r in records
        ]

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        branches = [torch.from_numpy(group[idx]) for group in self.groups]
        context = torch.from_numpy(self.contexts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        weight = torch.tensor(self.weights[idx], dtype=torch.float32)
        dataset_id = torch.tensor(self.dataset_ids[idx], dtype=torch.long)
        return branches, context, label, weight, dataset_id


def collate_fn(batch):
    branch_lists, contexts, labels, weights, dataset_ids = zip(*batch)
    n_branches = len(branch_lists[0])
    branches = [
        torch.stack([sample[group_idx] for sample in branch_lists])
        for group_idx in range(n_branches)
    ]
    return (
        branches,
        torch.stack(list(contexts)),
        torch.stack(list(labels)),
        torch.stack(list(weights)),
        torch.stack(list(dataset_ids)),
    )


In [ ]:
# =============================================================================
# Model


In [ ]:
# =============================================================================

class UncertaintyEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
        )
        self.mu_head = nn.Linear(hidden_dim, hidden_dim)
        self.logvar_head = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.shared(x)
        mu = self.mu_head(hidden)
        logvar = self.logvar_head(hidden).clamp(min=-10.0, max=10.0)
        return mu, logvar


class MetaUncertaintyGate(nn.Module):
    """Context-conditioned evidence gate for single-pass features."""

    SUPPORTED_KERNELS = ("softmax", "softplus", "relu")

    def __init__(
        self,
        branch_dims: Optional[List[int]] = None,
        context_dim: int = CONTEXT_DIM,
        hidden_dim: int = 32,
        dropout: float = 0.30,
        init_temperature: float = 1.0,
        attn_kernel: str = "softplus",
        branch_dropout_p: float = 0.30,
        min_branches_alive: int = 1,
        moe_logit_weight: float = 0.70,
        use_dataset_specific_head: bool = False,
        dataset_head_weight: float = 0.0,
        num_dataset_heads: int = 3,
    ) -> None:
        super().__init__()
        self.branch_dims = list(branch_dims or BRANCH_DIMS)
        self.evidence_branch_idxs = list(EVIDENCE_BRANCH_IDXS)
        self.n_evidence = len(self.evidence_branch_idxs)
        self.attn_kernel = attn_kernel
        self.branch_dropout_p = float(branch_dropout_p)
        self.min_branches_alive = max(1, int(min_branches_alive))
        self.moe_logit_weight = float(moe_logit_weight)
        self.use_dataset_specific_head = bool(use_dataset_specific_head)
        self.dataset_head_weight = float(dataset_head_weight)
        self.num_dataset_heads = int(num_dataset_heads)

        if self.attn_kernel not in self.SUPPORTED_KERNELS:
            raise ValueError(f"attn_kernel must be one of {self.SUPPORTED_KERNELS}")

        self.branch_encoders = nn.ModuleList(
            [UncertaintyEncoder(dim, hidden_dim, dropout) for dim in self.branch_dims]
        )
        self.branch_heads = nn.ModuleList(
            [nn.Linear(hidden_dim, 1) for _ in self.evidence_branch_idxs]
        )
        self.context_encoder = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.attention_net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.log_temperature = nn.Parameter(
            torch.tensor([math.log(max(init_temperature, 1e-3))], dtype=torch.float32)
        )
        self.failure_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        if self.use_dataset_specific_head:
            self.dataset_residual_head = nn.Sequential(
                nn.LayerNorm(hidden_dim * 2),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, self.num_dataset_heads),
            )
        else:
            self.dataset_residual_head = None

    def get_temperature(self) -> torch.Tensor:
        return torch.exp(self.log_temperature).clamp(min=0.05, max=5.0)

    def _kernel_attention(self, logits: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        scaled = logits / tau
        if self.attn_kernel == "softmax":
            return torch.softmax(scaled, dim=-1)
        if self.attn_kernel == "softplus":
            scores = F.softplus(scaled)
            return scores / scores.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        scores = torch.relu(scaled) + 1e-3
        return scores / scores.sum(dim=-1, keepdim=True).clamp_min(1e-12)

    def _apply_branch_dropout(self, weights: torch.Tensor) -> torch.Tensor:
        if (not self.training) or self.branch_dropout_p <= 0.0:
            return weights
        behavioral_global_idx = 2
        if behavioral_global_idx not in self.evidence_branch_idxs:
            return weights
        behavioral_pos = self.evidence_branch_idxs.index(behavioral_global_idx)
        if weights.shape[1] - 1 < self.min_branches_alive:
            return weights
        drop_mask = torch.rand(weights.shape[0], device=weights.device) < self.branch_dropout_p
        if not drop_mask.any():
            return weights
        keep = torch.ones_like(weights)
        keep[drop_mask, behavioral_pos] = 0.0
        dropped = weights * keep
        return dropped / dropped.sum(dim=-1, keepdim=True).clamp_min(1e-8)

    def forward(
        self,
        branches: List[torch.Tensor],
        context: torch.Tensor,
        dataset_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        ctx = self.context_encoder(context)

        mus: List[torch.Tensor] = []
        logvars: List[torch.Tensor] = []
        for encoder, x in zip(self.branch_encoders, branches):
            mu, logvar = encoder(x)
            mus.append(mu)
            logvars.append(logvar)

        attn_logits = torch.cat(
            [self.attention_net(torch.cat([mus[i], ctx], dim=-1)) for i in self.evidence_branch_idxs],
            dim=-1,
        )
        attn_weights = self._kernel_attention(attn_logits, self.get_temperature())
        attn_weights = self._apply_branch_dropout(attn_weights)

        branch_logits = torch.cat(
            [head(mus[i]) for head, i in zip(self.branch_heads, self.evidence_branch_idxs)],
            dim=-1,
        )
        moe_logit = (attn_weights * branch_logits).sum(dim=-1)

        eps = 1e-8
        evidence_mus = [mus[i] for i in self.evidence_branch_idxs]
        evidence_logvars = [logvars[i] for i in self.evidence_branch_idxs]
        precisions = [torch.exp(-lv) for lv in evidence_logvars]
        effective_precisions = [
            attn_weights[:, i:i + 1] * precisions[i] for i in range(self.n_evidence)
        ]
        total_precision = sum(effective_precisions) + eps
        fused_mu = sum(p * m for p, m in zip(effective_precisions, evidence_mus)) / total_precision
        fused_logvar = -torch.log(total_precision)
        fused_repr = torch.cat([fused_mu, fused_logvar], dim=-1)
        fusion_logit = self.failure_head(fused_repr).squeeze(-1)

        alpha = min(max(self.moe_logit_weight, 0.0), 1.0)
        logit = alpha * moe_logit + (1.0 - alpha) * fusion_logit

        if self.dataset_residual_head is not None and dataset_ids is not None:
            ds_ids = dataset_ids.to(logit.device).long()
            valid = (ds_ids >= 0) & (ds_ids < self.num_dataset_heads)
            safe_ids = ds_ids.clamp(min=0, max=max(self.num_dataset_heads - 1, 0))
            ds_logits = self.dataset_residual_head(fused_repr)
            residual = ds_logits.gather(1, safe_ids.view(-1, 1)).squeeze(-1)
            logit = logit + self.dataset_head_weight * residual * valid.float()

        p_fail = torch.sigmoid(logit)
        return p_fail, logit, attn_weights, fused_logvar, attn_logits, branch_logits


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class MetaGateLoss(nn.Module):
    def __init__(
        self,
        pos_weight: float,
        entropy_reg_weight: float = 0.02,
        branch_aux_weight: float = 0.20,
        cross_sample_diversity_weight: float = 0.10,
        group_loss_weight: float = 1.0,
        truthfulqa_rank_loss_weight: float = 0.0,
        truthfulqa_rank_temperature: float = 1.0,
        truthfulqa_dataset_id: int = DATASET_NAME_TO_ID["TruthfulQA"],
    ) -> None:
        super().__init__()
        self.pos_weight = torch.tensor([pos_weight], device=DEVICE)
        self.entropy_reg_weight = float(entropy_reg_weight)
        self.branch_aux_weight = float(branch_aux_weight)
        self.cross_sample_diversity_weight = float(cross_sample_diversity_weight)
        self.group_loss_weight = float(group_loss_weight)
        self.truthfulqa_rank_loss_weight = float(truthfulqa_rank_loss_weight)
        self.truthfulqa_rank_temperature = max(float(truthfulqa_rank_temperature), 1e-6)
        self.truthfulqa_dataset_id = int(truthfulqa_dataset_id)

    def _weighted_mean(
        self,
        loss_each: torch.Tensor,
        sample_weight: Optional[torch.Tensor],
    ) -> torch.Tensor:
        base = loss_each.mean()
        if sample_weight is None or self.group_loss_weight <= 0.0:
            return base
        weights = sample_weight.to(loss_each.device).float()
        weights = weights / weights.mean().clamp_min(1e-8)
        while weights.dim() < loss_each.dim():
            weights = weights.unsqueeze(-1)
        weighted = (loss_each * weights).mean()
        lam = min(max(self.group_loss_weight, 0.0), 1.0)
        return (1.0 - lam) * base + lam * weighted

    def _truthfulqa_rank_loss(
        self,
        logit: torch.Tensor,
        labels: torch.Tensor,
        dataset_ids: torch.Tensor,
    ) -> torch.Tensor:
        if self.truthfulqa_rank_loss_weight <= 0.0:
            return logit.new_tensor(0.0)
        tqa = dataset_ids.to(logit.device).long() == self.truthfulqa_dataset_id
        pos = logit[tqa & (labels > 0.5)]
        neg = logit[tqa & (labels <= 0.5)]
        if pos.numel() == 0 or neg.numel() == 0:
            return logit.new_tensor(0.0)
        margins = (pos[:, None] - neg[None, :]) / self.truthfulqa_rank_temperature
        return F.softplus(-margins).mean()

    def forward(
        self,
        logit: torch.Tensor,
        labels: torch.Tensor,
        attn_weights: torch.Tensor,
        branch_logits: torch.Tensor,
        sample_weight: torch.Tensor,
        dataset_ids: torch.Tensor,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        bce_each = F.binary_cross_entropy_with_logits(
            logit,
            labels,
            pos_weight=self.pos_weight,
            reduction="none",
        )
        bce = self._weighted_mean(bce_each, sample_weight)

        if self.branch_aux_weight > 0.0:
            branch_targets = labels[:, None].expand_as(branch_logits)
            branch_each = F.binary_cross_entropy_with_logits(
                branch_logits,
                branch_targets,
                pos_weight=self.pos_weight,
                reduction="none",
            )
            branch_aux = self._weighted_mean(branch_each, sample_weight)
        else:
            branch_aux = logit.new_tensor(0.0)

        eps = 1e-8
        entropy = -(attn_weights * (attn_weights + eps).log()).sum(dim=-1).mean()
        if self.cross_sample_diversity_weight > 0.0 and attn_weights.shape[0] > 1:
            diversity = attn_weights.std(dim=0).mean()
        else:
            diversity = logit.new_tensor(0.0)

        rank_loss = self._truthfulqa_rank_loss(logit, labels, dataset_ids)
        loss = (
            bce
            + self.branch_aux_weight * branch_aux
            + self.truthfulqa_rank_loss_weight * rank_loss
            + self.entropy_reg_weight * entropy
            - self.cross_sample_diversity_weight * diversity
        )
        return loss, {
            "bce": float(bce.detach().cpu().item()),
            "branch_aux": float(branch_aux.detach().cpu().item()),
            "entropy": float(entropy.detach().cpu().item()),
            "diversity": float(diversity.detach().cpu().item()),
            "rank": float(rank_loss.detach().cpu().item()),
            "loss": float(loss.detach().cpu().item()),
        }


In [ ]:
# =============================================================================
# Training and inference


In [ ]:
# =============================================================================

def compute_pos_weight(records: Sequence[Dict[str, Any]]) -> float:
    labels = np.asarray([int(r["label"]) for r in records], dtype=int)
    n_pos = int((labels == 1).sum())
    n_neg = int((labels == 0).sum())
    return float(n_neg / max(n_pos, 1))


def train_epoch(
    model: MetaUncertaintyGate,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: MetaGateLoss,
) -> float:
    model.train()
    total = 0.0
    for branches, context, labels, sample_weight, dataset_ids in loader:
        branches = [x.to(DEVICE) for x in branches]
        context = context.to(DEVICE)
        labels = labels.to(DEVICE)
        sample_weight = sample_weight.to(DEVICE)
        dataset_ids = dataset_ids.to(DEVICE)

        optimizer.zero_grad()
        _, logit, attn_weights, _, _, branch_logits = model(branches, context, dataset_ids)
        loss, _ = criterion(logit, labels, attn_weights, branch_logits, sample_weight, dataset_ids)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += float(loss.detach().cpu().item())
    return total / max(len(loader), 1)


@torch.no_grad()
def eval_epoch(
    model: MetaUncertaintyGate,
    loader: DataLoader,
    criterion: MetaGateLoss,
) -> Tuple[float, float]:
    model.eval()
    total = 0.0
    scores: List[float] = []
    labels_all: List[float] = []

    for branches, context, labels, sample_weight, dataset_ids in loader:
        branches = [x.to(DEVICE) for x in branches]
        context = context.to(DEVICE)
        labels = labels.to(DEVICE)
        sample_weight = sample_weight.to(DEVICE)
        dataset_ids = dataset_ids.to(DEVICE)

        p_fail, logit, attn_weights, _, _, branch_logits = model(branches, context, dataset_ids)
        loss, _ = criterion(logit, labels, attn_weights, branch_logits, sample_weight, dataset_ids)
        total += float(loss.detach().cpu().item())
        scores.extend(p_fail.detach().cpu().numpy().tolist())
        labels_all.extend(labels.detach().cpu().numpy().tolist())

    return total / max(len(loader), 1), binary_auroc(np.asarray(labels_all), np.asarray(scores))


@torch.no_grad()
def get_predictions(
    model: MetaUncertaintyGate,
    records: Sequence[Dict[str, Any]],
    normalizer: FeatureNormalizer,
    batch_size: int = 512,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if not records:
        return np.array([]), np.array([]), np.array([]), np.array([])

    dataset = MetaGateDataset(records, normalizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    model.eval()

    all_scores: List[np.ndarray] = []
    all_attn: List[np.ndarray] = []
    all_logvar: List[np.ndarray] = []

    for branches, context, _, _, dataset_ids in loader:
        branches = [x.to(DEVICE) for x in branches]
        context = context.to(DEVICE)
        dataset_ids = dataset_ids.to(DEVICE)
        p_fail, _, attn_weights, fused_logvar, _, _ = model(branches, context, dataset_ids)
        all_scores.append(p_fail.detach().cpu().numpy())
        all_attn.append(attn_weights.detach().cpu().numpy())
        all_logvar.append(fused_logvar.detach().cpu().numpy())

    y_true = np.asarray([int(r["label"]) for r in records], dtype=int)
    y_score = np.concatenate(all_scores)
    attn = np.concatenate(all_attn)
    fused_logvar = np.concatenate(all_logvar)
    return y_true, y_score, attn, fused_logvar


def validation_selection_score(
    val_records: Sequence[Dict[str, Any]],
    y_true: np.ndarray,
    y_score: np.ndarray,
) -> float:
    pooled = compute_all_metrics(y_true, y_score, threshold=0.5)
    by_dataset = compute_group_metrics(val_records, y_true, y_score, "dataset", threshold=0.5)
    dataset_aurocs = [m["auroc"] for m in by_dataset.values()]
    macro_auroc = mean_finite(dataset_aurocs)
    worst_auroc = worst_finite(dataset_aurocs)

    def finite_or_zero(value: float) -> float:
        value = float(value)
        return value if math.isfinite(value) else 0.0

    return (
        0.35 * finite_or_zero(pooled["auroc"])
        + 0.30 * finite_or_zero(macro_auroc)
        + 0.30 * finite_or_zero(worst_auroc)
        + 0.05 * finite_or_zero(pooled["auprc"])
    )


def train_model(
    train_records: List[Dict[str, Any]],
    val_records: List[Dict[str, Any]],
    model_cfg: Dict[str, Any],
    train_cfg: Dict[str, Any],
) -> Tuple[MetaUncertaintyGate, FeatureNormalizer, Dict[str, Any]]:
    normalizer = FeatureNormalizer().fit(train_records)
    train_ds = MetaGateDataset(train_records, normalizer)
    val_ds = MetaGateDataset(val_records, normalizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=train_cfg["batch_size"],
        shuffle=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=max(1, min(len(val_ds), train_cfg.get("eval_batch_size", 1024))),
        shuffle=False,
        collate_fn=collate_fn,
    )

    model = MetaUncertaintyGate(**model_cfg).to(DEVICE)
    criterion = MetaGateLoss(
        pos_weight=compute_pos_weight(train_records),
        entropy_reg_weight=train_cfg["entropy_reg"],
        branch_aux_weight=train_cfg["branch_aux_weight"],
        cross_sample_diversity_weight=train_cfg["cross_sample_diversity_weight"],
        group_loss_weight=train_cfg["group_loss_weight"],
        truthfulqa_rank_loss_weight=train_cfg["truthfulqa_rank_loss_weight"],
        truthfulqa_rank_temperature=train_cfg["truthfulqa_rank_temperature"],
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=train_cfg["lr"],
        weight_decay=train_cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        patience=max(train_cfg["patience"] // 3, 1),
        factor=0.5,
    )

    history: Dict[str, Any] = {
        "train_loss": [],
        "val_loss": [],
        "val_auroc": [],
        "val_select": [],
        "temperature": [],
    }
    best_score = -float("inf")
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_epoch = 0
    patience_count = 0

    print("\n" + "=" * 72)
    print("Training MetaUncertaintyGate on single-pass features")
    print("=" * 72)
    print(f"  Device      : {DEVICE}")
    print(f"  Parameters  : {count_parameters(model)}")
    print(f"  Train/Val   : {len(train_records)} / {len(val_records)}")
    print("=" * 72)

    for epoch in range(1, train_cfg["epochs"] + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_auroc = eval_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        y_val, s_val, _, _ = get_predictions(
            model,
            val_records,
            normalizer,
            batch_size=train_cfg.get("eval_batch_size", 1024),
        )
        val_select = validation_selection_score(val_records, y_val, s_val)
        tau = float(model.get_temperature().detach().cpu().item())

        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))
        history["val_auroc"].append(float(val_auroc))
        history["val_select"].append(float(val_select))
        history["temperature"].append(float(tau))

        if val_select > best_score + 1e-6:
            best_score = float(val_select)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_count = 0
        else:
            patience_count += 1

        if epoch == 1 or epoch % train_cfg["log_every"] == 0:
            print(
                f"  Epoch {epoch:>3} "
                f"train={train_loss:.4f} val={val_loss:.4f} "
                f"val_auroc={val_auroc:.4f} select={val_select:.4f} tau={tau:.3f}"
            )

        if patience_count >= train_cfg["patience"]:
            print(f"  Early stop at epoch {epoch}; best epoch={best_epoch}, score={best_score:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history["best_epoch"] = int(best_epoch)
    history["best_val_select"] = float(best_score)
    return model, normalizer, history


In [ ]:
# =============================================================================
# Experiment runner


In [ ]:
# =============================================================================

def evaluate_strategies(
    records: Sequence[Dict[str, Any]],
    y_true: np.ndarray,
    y_score: np.ndarray,
    fixed_threshold: float,
    global_threshold: float,
    per_dataset_thresholds: Dict[str, Dict[str, Any]],
    per_task_thresholds: Dict[str, Dict[str, Any]],
    headline_strategy: str,
) -> Tuple[Dict[str, Dict[str, Any]], np.ndarray, np.ndarray, str]:
    strategies = {
        "fixed": ("fixed", fixed_threshold, None, None),
        "global": ("global tuned", global_threshold, None, None),
        "per_dataset": ("per-dataset tuned", global_threshold, per_dataset_thresholds, "dataset"),
        "per_task": ("per-task tuned", global_threshold, per_task_thresholds, "task_type"),
    }

    results: Dict[str, Dict[str, Any]] = {}
    headline_pred = np.zeros(len(records), dtype=int)
    headline_thresholds = np.full(len(records), fixed_threshold)
    headline_label = "fixed"

    for key, (label, fallback, group_thresholds, group_key) in strategies.items():
        if group_thresholds is None:
            thresholds = np.full(len(records), float(fallback))
            pred = (y_score >= float(fallback)).astype(int)
        else:
            pred, thresholds = apply_group_thresholds(
                records,
                y_score,
                group_thresholds,
                str(group_key),
                float(fallback),
            )
        metrics = classification_from_preds(y_true, pred)
        metrics["label"] = label
        metrics["strategy"] = key
        metrics["threshold_mean"] = float(np.mean(thresholds)) if len(thresholds) else float("nan")
        results[key] = metrics
        if key == headline_strategy:
            headline_pred = pred
            headline_thresholds = thresholds
            headline_label = label

    return results, headline_pred, headline_thresholds, headline_label


def attention_summary(attn: np.ndarray) -> Dict[str, Dict[str, float]]:
    out: Dict[str, Dict[str, float]] = {}
    if attn.size == 0:
        return out
    for i, name in enumerate(EVIDENCE_BRANCH_NAMES):
        out[name] = {
            "mean": float(attn[:, i].mean()),
            "std": float(attn[:, i].std()),
            "min": float(attn[:, i].min()),
            "max": float(attn[:, i].max()),
        }
    return out


def print_strategy_table(strategy_results: Dict[str, Dict[str, Any]], headline: str) -> None:
    print("\nTest classification strategy comparison")
    print("  strategy                  Acc     Prec    Rec     F1      TP   FP   TN   FN")
    print("  " + "-" * 82)
    for key in ["fixed", "global", "per_dataset", "per_task"]:
        item = strategy_results[key]
        marker = "  headline" if key == headline else ""
        print(
            f"  {item['label']:<22} "
            f"{item['accuracy']:>6.3f}  {item['precision']:>6.3f}  "
            f"{item['recall']:>6.3f}  {item['f1']:>6.3f}  "
            f"{item['tp']:>4} {item['fp']:>4} {item['tn']:>4} {item['fn']:>4}"
            f"{marker}"
        )


def write_predictions_csv(
    path: str,
    records: Sequence[Dict[str, Any]],
    y_true: np.ndarray,
    y_score: np.ndarray,
    y_pred: np.ndarray,
    thresholds: np.ndarray,
    attn: np.ndarray,
) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fieldnames = [
        "sample_id",
        "dataset",
        "task_type",
        "label_failure",
        "p_failure",
        "prediction_failure",
        "threshold",
    ] + [f"attn_{name}" for name in EVIDENCE_BRANCH_NAMES]

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for i, record in enumerate(records):
            row = {
                "sample_id": record.get("sample_id", ""),
                "dataset": record.get("dataset", ""),
                "task_type": record.get("task_type", ""),
                "label_failure": int(y_true[i]),
                "p_failure": float(y_score[i]),
                "prediction_failure": int(y_pred[i]),
                "threshold": float(thresholds[i]),
            }
            for j, name in enumerate(EVIDENCE_BRANCH_NAMES):
                row[f"attn_{name}"] = float(attn[i, j]) if attn.size else float("nan")
            writer.writerow(row)


def run_single_time(args: argparse.Namespace) -> Dict[str, Any]:
    global DEVICE
    if args.device == "cpu":
        DEVICE = torch.device("cpu")
    elif args.device == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("--device cuda requested but CUDA is not available.")
        DEVICE = torch.device("cuda")
    else:
        DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    set_seed(args.seed)
    print_feature_schema()

    records = load_jsonl(
        args.data_path,
        label_source=args.label_source,
        label_mode=args.label_mode,
    )
    if len(records) == 0:
        raise ValueError("No records loaded. Check --data-path and label settings.")
    schema_report = validate_features(records, allow_missing=args.allow_missing_features)
    print_data_stats(records)

    train_records, val_records, test_records = stratified_split_records(
        records,
        train_ratio=args.train_ratio,
        val_ratio=args.val_ratio,
        seed=args.seed,
        stratify_keys=("dataset", "label"),
    )
    split_info = split_summary(train_records, val_records, test_records)
    print("\nSplit summary")
    print(json.dumps(split_info, ensure_ascii=False, indent=2))

    model_cfg = {
        "branch_dims": BRANCH_DIMS,
        "context_dim": CONTEXT_DIM,
        "hidden_dim": args.hidden_dim,
        "dropout": args.dropout,
        "init_temperature": args.init_temperature,
        "attn_kernel": args.attn_kernel,
        "branch_dropout_p": args.branch_dropout_p,
        "min_branches_alive": args.min_branches_alive,
        "moe_logit_weight": args.moe_logit_weight,
        "use_dataset_specific_head": args.use_dataset_head,
        "dataset_head_weight": args.dataset_head_weight,
        "num_dataset_heads": len(DATASET_NAME_TO_ID),
    }
    train_cfg = {
        "batch_size": args.batch_size,
        "eval_batch_size": args.eval_batch_size,
        "epochs": args.epochs,
        "lr": args.lr,
        "weight_decay": args.weight_decay,
        "patience": args.patience,
        "log_every": args.log_every,
        "entropy_reg": args.entropy_reg,
        "branch_aux_weight": args.branch_aux_weight,
        "cross_sample_diversity_weight": args.cross_sample_diversity_weight,
        "group_loss_weight": args.group_loss_weight,
        "truthfulqa_rank_loss_weight": args.truthfulqa_rank_loss_weight,
        "truthfulqa_rank_temperature": args.truthfulqa_rank_temperature,
    }

    model, normalizer, history = train_model(train_records, val_records, model_cfg, train_cfg)

    y_val, s_val, _, _ = get_predictions(model, val_records, normalizer, args.eval_batch_size)
    global_thr, global_score = tune_threshold(
        y_val,
        s_val,
        metric=args.threshold_metric,
        fallback=args.fixed_threshold,
        min_pred_pos_rate=args.threshold_min_pred_pos_rate,
        max_pred_pos_rate=args.threshold_max_pred_pos_rate,
        avoid_degenerate=not args.allow_degenerate_thresholds,
    )
    per_dataset_thrs = tune_group_thresholds(
        val_records,
        y_val,
        s_val,
        "dataset",
        metric=args.threshold_metric,
        fallback=global_thr,
        min_pred_pos_rate=args.threshold_min_pred_pos_rate,
        max_pred_pos_rate=args.threshold_max_pred_pos_rate,
        avoid_degenerate=not args.allow_degenerate_thresholds,
    )
    per_task_thrs = tune_group_thresholds(
        val_records,
        y_val,
        s_val,
        "task_type",
        metric=args.threshold_metric,
        fallback=global_thr,
        min_pred_pos_rate=args.threshold_min_pred_pos_rate,
        max_pred_pos_rate=args.threshold_max_pred_pos_rate,
        avoid_degenerate=not args.allow_degenerate_thresholds,
    )

    print(f"\nGlobal tuned threshold: {global_thr:.4f} ({args.threshold_metric}={global_score:.4f})")
    print_threshold_table(per_dataset_thrs, "Per-dataset thresholds from validation")
    print_threshold_table(per_task_thrs, "Per-task thresholds from validation")

    y_test, s_test, attn_test, fused_logvar = get_predictions(
        model,
        test_records,
        normalizer,
        args.eval_batch_size,
    )
    ranking = compute_all_metrics(y_test, s_test, threshold=args.fixed_threshold)
    strategy_results, headline_pred, headline_thresholds, headline_label = evaluate_strategies(
        test_records,
        y_test,
        s_test,
        fixed_threshold=args.fixed_threshold,
        global_threshold=global_thr,
        per_dataset_thresholds=per_dataset_thrs,
        per_task_thresholds=per_task_thrs,
        headline_strategy=args.headline_strategy,
    )
    within_dataset = compute_group_metrics(
        test_records,
        y_test,
        s_test,
        "dataset",
        threshold=global_thr,
        group_thresholds=per_dataset_thrs,
    )
    within_task = compute_group_metrics(
        test_records,
        y_test,
        s_test,
        "task_type",
        threshold=global_thr,
        group_thresholds=per_task_thrs,
    )

    print("\nTest ranking metrics")
    print(f"  AUROC : {ranking['auroc']:.4f}")
    print(f"  AUPRC : {ranking['auprc']:.4f}")
    print(f"  Brier : {ranking['brier']:.4f}")
    print(f"  ECE   : {ranking['ece']:.4f}")
    print_strategy_table(strategy_results, args.headline_strategy)
    print_group_metrics_table(within_dataset, "Test within-dataset metrics")
    print_group_metrics_table(within_task, "Test within-task metrics")

    attn_summary = attention_summary(attn_test)
    print("\nAttention summary on test set")
    for name, item in attn_summary.items():
        print(
            f"  {name:<10} mean={item['mean']:.3f} std={item['std']:.3f} "
            f"min={item['min']:.3f} max={item['max']:.3f}"
        )

    macro_dataset_auroc = mean_finite(m["auroc"] for m in within_dataset.values())
    worst_dataset_auroc = worst_finite(m["auroc"] for m in within_dataset.values())
    macro_dataset_f1 = mean_finite(m["f1"] for m in within_dataset.values())
    worst_dataset_f1 = worst_finite(m["f1"] for m in within_dataset.values())

    results = {
        "script": "single time.py",
        "data_path": args.data_path,
        "label_source": args.label_source,
        "label_mode": args.label_mode,
        "feature_schema": {
            "branch_names": BRANCH_NAMES,
            "branch_dims": BRANCH_DIMS,
            "branch_groups": ALL_BRANCH_GROUPS,
            "deleted_k5_features": K5_FEATURES_DELETED_FOR_SINGLE_PASS,
            "validation": schema_report,
        },
        "split_summary": split_info,
        "model_cfg": model_cfg,
        "train_cfg": train_cfg,
        "history": history,
        "test_ranking": ranking,
        "thresholds": {
            "fixed": args.fixed_threshold,
            "global": global_thr,
            "global_score": global_score,
            "per_dataset": per_dataset_thrs,
            "per_task": per_task_thrs,
            "headline_strategy": args.headline_strategy,
            "headline_label": headline_label,
        },
        "strategy_results": strategy_results,
        "within_dataset": within_dataset,
        "within_task": within_task,
        "summary": {
            "macro_dataset_auroc": macro_dataset_auroc,
            "worst_dataset_auroc": worst_dataset_auroc,
            "macro_dataset_f1": macro_dataset_f1,
            "worst_dataset_f1": worst_dataset_f1,
        },
        "attention_summary": attn_summary,
    }

    os.makedirs(args.output_dir, exist_ok=True)
    metrics_path = os.path.join(args.output_dir, "single_time_metrics.json")
    pred_path = os.path.join(args.output_dir, "single_time_predictions.csv")
    save_json(results, metrics_path)
    write_predictions_csv(
        pred_path,
        test_records,
        y_test,
        s_test,
        headline_pred,
        headline_thresholds,
        attn_test,
    )

    if not args.no_save_model:
        model_path = os.path.join(args.output_dir, "single_time_model.pt")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "normalizer": normalizer.state_dict(),
                "model_cfg": model_cfg,
                "branch_groups": ALL_BRANCH_GROUPS,
                "dataset_name_to_id": DATASET_NAME_TO_ID,
            },
            model_path,
        )
        print(f"\nSaved model checkpoint: {model_path}")

    print(f"Saved metrics: {metrics_path}")
    print(f"Saved predictions: {pred_path}")
    return results


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Train/evaluate MetaUncertaintyGate on single-pass v13 features."
    )
    parser.add_argument("--data-path", default=DEFAULT_DATA_PATH)
    parser.add_argument("--output-dir", default=DEFAULT_OUTPUT_DIR)
    parser.add_argument(
        "--label-source",
        default="single_is_failure",
        choices=["auto", "single_is_failure", "single_is_correct", "label"],
        help="Default uses single_is_failure, i.e. 1=failure.",
    )
    parser.add_argument(
        "--label-mode",
        default="auto",
        choices=["auto", "correct_is_1", "failure_is_1"],
        help="Only used when --label-source label.",
    )
    parser.add_argument("--allow-missing-features", action="store_true")

    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--device", choices=["auto", "cpu", "cuda"], default="auto")
    parser.add_argument("--train-ratio", type=float, default=0.70)
    parser.add_argument("--val-ratio", type=float, default=0.15)

    parser.add_argument("--epochs", type=int, default=150)
    parser.add_argument("--patience", type=int, default=25)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--eval-batch-size", type=int, default=1024)
    parser.add_argument("--hidden-dim", type=int, default=32)
    parser.add_argument("--dropout", type=float, default=0.30)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--log-every", type=int, default=20)

    parser.add_argument("--init-temperature", type=float, default=1.0)
    parser.add_argument(
        "--attn-kernel",
        choices=["softmax", "softplus", "relu"],
        default="softplus",
    )
    parser.add_argument("--branch-dropout-p", type=float, default=0.30)
    parser.add_argument("--min-branches-alive", type=int, default=1)
    parser.add_argument("--moe-logit-weight", type=float, default=0.70)
    parser.add_argument("--entropy-reg", type=float, default=0.02)
    parser.add_argument("--branch-aux-weight", type=float, default=0.20)
    parser.add_argument("--cross-sample-diversity-weight", type=float, default=0.10)
    parser.add_argument("--group-loss-weight", type=float, default=1.0)
    parser.add_argument("--truthfulqa-rank-loss-weight", type=float, default=0.0)
    parser.add_argument("--truthfulqa-rank-temperature", type=float, default=1.0)
    parser.add_argument("--use-dataset-head", action="store_true")
    parser.add_argument("--dataset-head-weight", type=float, default=0.10)

    parser.add_argument("--fixed-threshold", type=float, default=0.5)
    parser.add_argument(
        "--threshold-metric",
        choices=["f1", "accuracy", "balanced_acc", "youden"],
        default="f1",
    )
    parser.add_argument(
        "--headline-strategy",
        choices=["fixed", "global", "per_dataset", "per_task"],
        default="per_dataset",
    )
    parser.add_argument("--threshold-min-pred-pos-rate", type=float, default=0.02)
    parser.add_argument("--threshold-max-pred-pos-rate", type=float, default=0.85)
    parser.add_argument("--allow-degenerate-thresholds", action="store_true")
    parser.add_argument("--no-save-model", action="store_true")
    return parser


def main() -> None:
    parser = build_arg_parser()
    args = parser.parse_args()
    run_single_time(args)


## Run On Colab

Edit the configuration below if you want shorter smoke tests or different hyperparameters. By default, the notebook downloads `single time.jsonl` from your GitHub repo into `/content/`.


In [ ]:
# Download data from GitHub and run the experiment.
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/hsun0912/5703sunhao/main/data/single%20time.jsonl"
DATA_PATH = "/content/single time.jsonl" if os.path.isdir("/content") else "single time.jsonl"
OUTPUT_DIR = "/content/single_time_outputs" if os.path.isdir("/content") else os.path.join("outputs", "single_time")

if not os.path.exists(DATA_PATH):
    print(f"Downloading data to {DATA_PATH} ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
else:
    print(f"Using existing data file: {DATA_PATH}")

parser = build_arg_parser()
args = parser.parse_args([])  # Important for notebooks/Colab: ignore kernel CLI args.
args.data_path = DATA_PATH
args.output_dir = OUTPUT_DIR
args.device = "auto"

# Main knobs. For a quick smoke test, set args.epochs = 3.
args.epochs = 150
args.patience = 25
args.batch_size = 64
args.hidden_dim = 32
args.headline_strategy = "per_dataset"

results = run_single_time(args)
results["summary"]


## Output Files

After the run, Colab writes metrics, predictions, and the model checkpoint under `OUTPUT_DIR`.


In [ ]:
print("OUTPUT_DIR =", OUTPUT_DIR)
print("Files:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, name)
    print(f"  {name:32s} {os.path.getsize(path):>10} bytes")
